# 03 - AQUAVIEW data via STAC (R)

**File:** `notebooks/03_aquaview_stac_r.ipynb`

**What this does:** Searches the public AQUAVIEW catalogue, picks a glider deployment, downloads a slice of it into a data frame and plots it.

**How to run it:** Open this file in JupyterLab, check that the kernel shown in the
top-right corner says **R**, then choose *Run > Run All Cells*.

**Inputs:** none -- reads from the public AQUAVIEW API over the internet

**Outputs:** a data frame of glider measurements and a temperature-vs-depth plot

## What is STAC?

AQUAVIEW publishes its catalogue as a **STAC** API (SpatioTemporal Asset
Catalog) -- a shared standard for describing data that has a location and a
time. The useful part: it is ordinary JSON over ordinary web requests, so no
special library is needed.

Three ideas cover almost everything:

| Term | Meaning |
|---|---|
| **Collection** | A group of related data, e.g. `IOOS` = the IOOS Glider Data Assembly Center. |
| **Item** | One thing inside a collection, e.g. a single glider deployment. |
| **Asset** | A downloadable file attached to an item, e.g. that deployment as CSV. |

The catalogue is **public** -- no account, no API key, no login.

In [ ]:
library(jsonlite)

# The AQUAVIEW catalogue. No key or login needed.
BASE <- "https://service.aquaview.org/stac"

# simplifyVector = FALSE keeps the result as plain nested lists, which is
# easier to reason about than jsonlite's automatic conversion.
collections <- fromJSON(paste0(BASE, "/collections"), simplifyVector = FALSE)$collections

cat(length(collections), "collections available. The first 10:\n\n")

for (collection in collections[1:10]) {
  cat(sprintf("  %-16s %s\n", collection$id, substr(collection$title, 1, 58)))
}

## 2. Searching for items

`IOOS` is the IOOS Glider Data Assembly Center -- autonomous underwater gliders.
Let us look around **Hawaiʻi** -- the same waters the hackathon glider data
comes from, so the two fit together.

In [ ]:
search_url <- paste0(
  BASE, "/search",
  "?collections=IOOS",
  "&bbox=-161,18,-154,23",   # west, south, east, north
  "&limit=5"
)

results <- fromJSON(URLencode(search_url), simplifyVector = FALSE)

cat("items matching the search:", results$numberMatched, "\n")
cat("returned in this page:    ", length(results$features), "\n\n")

for (feature in results$features) cat(" ", feature$id, "\n")

### Reading the search settings

- `collections` -- which collection to look in. Leave it out to search all 100.
- `bbox` -- a box on the map, given as `west,south,east,north`. The values above
  cover the Hawaiian Islands. The Glider Rodeo waypoints sit just off Oʻahu,
  around 21.2 N, 158.2 W, well inside that box.
- `limit` -- how many items to return at once.

You can also pass `datetime` to restrict the time range, like
`2016-01-01T00:00:00Z/2017-12-31T23:59:59Z`.

> **Watch the order in `bbox`.** It is longitude first, then latitude
> (`west,south,east,north`). Getting this backwards is the most common mistake,
> and it usually returns nothing at all rather than an error.

## 3. Looking at one item

In [ ]:
item <- results$features[[1]]

cat("id:        ", item$id, "\n")
cat("collection:", item$collection, "\n")

# ERDDAP offers the same data in many formats, so just count them and
# show a few rather than printing all of them.
cat("assets:    ", length(item$assets), "formats available\n")
cat("            e.g.", paste(sort(names(item$assets))[1:10], collapse = ", "), "\n")

### From item to actual data

The item above is only a *description*. The real data lives at one of the URLs
in `assets`. We want the `csv` one.

These particular CSV links point at **ERDDAP**, a data server widely used in
ocean science. Two practical things follow:

> **Always ask for a time range.** A full glider deployment can be hundreds of
> megabytes, and without a time limit you download all of it. Adding
> `?&time>=...&time<=...` to the URL asks the server to send just that slice.

For these glider items the deployment start date is written into the item id
(`sg626-`**`20250729`**`T1452`), so the cell below pulls the date out of the
id and asks for that first day -- a day we know has data in it.

In [ ]:
csv_url <- item$assets$csv$href

# Pull the deployment start date out of the item id, e.g. "...-20250729T1452".
day <- substr(regmatches(item$id, regexpr("-[0-9]{8}T", item$id)), 2, 9)
start <- paste(substr(day, 1, 4), substr(day, 5, 6), substr(day, 7, 8), sep = "-")

data_url <- paste0(csv_url, "?&time>=", start, "&time<=", start, "T23:59:59Z")

cat("deployment started:", start, "\n")
cat("downloading:", data_url, "\n")

### One quirk: ERDDAP sends two header rows

An ERDDAP CSV has the column *names* on line 1 and the column *units* on line 2,
with the data starting on line 3. That second line has to be skipped, or every
number is read as text and no arithmetic works.

In [ ]:
# Read the raw lines, drop line 2 (the units), then parse what is left.
csv_lines <- readLines(url(data_url), warn = FALSE)
glider <- read.csv(text = paste(csv_lines[-2], collapse = "\n"))

cat(nrow(glider), "measurements,", ncol(glider), "columns\n")
head(glider[, c("time", "latitude", "longitude", "depth", "temperature")])

## 4. A quick look at the data

A glider dives up and down as it travels, so plotting temperature against depth
shows the shape of the water column. Depth increases downwards, so the axis is
flipped to match how we picture the ocean.

In [ ]:
library(ggplot2)

options(repr.plot.width = 5, repr.plot.height = 6)

ggplot(glider, aes(x = temperature, y = depth)) +
  geom_point(size = 0.6, alpha = 0.4) +
  scale_y_reverse() +
  labs(
    title = item$id,
    subtitle = start,
    x = "Temperature (C)",
    y = "Depth (m)"
  ) +
  theme_minimal()

## Done

Next: **`04_glider_satellite_r.ipynb`**, a longer real-world workflow comparing a
glider track against satellite data.

After that, start building your own thing -- see the [main README](../README.md).